# Training gradient boosting model for enzyme-substrate pair prediction with ESM-1b-vectors

### 1. Loading and preprocessing data for model training and evaluation
### 2. Hyperparameter optimization using a 5-fold cross-validation (CV)
### 3. Training and validating the final model

In [1]:
import pandas as pd
import numpy as np
import random
import pickle
import sys
import os
import logging
from os.path import join
from sklearn.model_selection import KFold
#from hyperopt import fmin, tpe, hp, Trials, rand
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.metrics import matthews_corrcoef

sys.path.append('./additional_code')
#from data_preprocessing import *

CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)

/home/hanxudong/Repositories/ESP/notebooks_and_code


## 1. Loading and preprocessing data for model training and evaluation

### (a) Loading data:

In [2]:
df_test  = pd.read_pickle(join(CURRENT_DIR, ".." ,"data","splits", "df_test_with_ESM1b_ts.pkl"))
df_test = df_test.loc[df_test["ESM1b_ts"] != ""]
df_test.reset_index(inplace = True, drop = True)

df_Mou  = pd.read_pickle(join(CURRENT_DIR, ".." ,"data", "Mou_data", "Mou_df.pkl"))
df_Mou = df_Mou.loc[df_test["ESM1b_ts"] != ""]
df_Mou.reset_index(inplace = True, drop = True)


/home/hanxudong/miniconda3/envs/esp/lib/python3.8/site-packages/pandas/core/ops/array_ops.py:82: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  result = libops.scalar_compare(x.ravel(), y, op)
/home/hanxudong/miniconda3/envs/esp/lib/python3.8/site-packages/pandas/core/ops/array_ops.py:82: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  result = libops.scalar_compare(x.ravel(), y, op)


Loading new dataset:

In [3]:
def create_input_and_output_data(df):
    X = ();
    y = ();
    
    for ind in df.index:
        emb = df["ESM1b_ts"][ind]
        ecfp = np.array(list(df["ECFP"][ind])).astype(int)
                
        X = X +(np.concatenate([ecfp, emb]), );
        y = y + (df["Binding"][ind], );

    return(np.array(X),np.array(y))

feature_names =  ["ECFP_" + str(i) for i in range(1024)]
feature_names = feature_names + ["ESM1b_ts_" + str(i) for i in range(1280)]


### Adding Mou et al. data to the training set:

In [4]:

df_train = pd.read_pickle(join(CURRENT_DIR, ".." ,"data",
                               "splits", "df_train_with_ESM1b_ts.pkl"))
df_train = df_train.loc[df_train["ESM1b_ts"] != ""]
df_train.reset_index(inplace = True, drop = True)


train_X, train_y =  create_input_and_output_data(df = df_train)
test_X, test_y =  create_input_and_output_data(df = df_test)

df_test_new = df_Mou.copy()
df_test_new["Binding"] = [y > 2 for y in df_test_new["activity"]]
test_new_X, test_new_y =  create_input_and_output_data(df = df_test_new)



from sklearn.model_selection import train_test_split
#same split as in Mou et al paper:
X_train_Mou, X_test_Mou, y_train_Mou, y_test_Mou = train_test_split(test_new_X, test_new_y,
                                                                    test_size = 0.20, random_state = 888)

train_X = np.concatenate([train_X, X_train_Mou])
train_y = np.concatenate([train_y, y_train_Mou])


/home/hanxudong/miniconda3/envs/esp/lib/python3.8/site-packages/pandas/core/ops/array_ops.py:82: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  result = libops.scalar_compare(x.ravel(), y, op)


In [5]:
param = {'learning_rate': 0.31553117247348733,
         'max_delta_step': 1.7726044219753656,
         'max_depth': 10,
         'min_child_weight': 1.3845040588450772,
         'num_rounds': 342.68325188584106,
         'reg_alpha': 0.531395259755843,
         'reg_lambda': 3.744980563764689,
         'weight': 0.26187490421514203}



num_round = param["num_rounds"]
#param["tree_method"] = "gpu_hist"
#param["sampling_method"] = "gradient_based"
param['objective'] = 'binary:logistic'

weights1 = np.array([param["weight"] if binding == 0 else 1.0 for binding in df_train["Binding"]])
weights2 = np.array([param["weight"] if binding == 0 else 1.0 for binding in y_train_Mou])

weights = np.concatenate([weights1, weights2])


del param["num_rounds"]
del param["weight"]

dtrain = xgb.DMatrix(np.array(train_X), weight = weights, label = np.array(train_y),
            feature_names= feature_names)
dtest = xgb.DMatrix(np.array(test_X), label = np.array(test_y),
                    feature_names= feature_names)

dtest_new = xgb.DMatrix(np.array(X_test_Mou), label = np.array(y_test_Mou), feature_names= feature_names)

evallist = [(dtest_new, 'eval'), (dtrain, 'train')]

bst = xgb.train(param,  dtrain, int(num_round),evallist, verbose_eval=1)
y_test_pred = np.round(bst.predict(dtest))
acc_test = np.mean(y_test_pred == np.array(test_y))
roc_auc = roc_auc_score(np.array(test_y), bst.predict(dtest))

print("Accuracy on test set: %s, ROC-AUC score for test set: %s"  % (acc_test, roc_auc))


y_test_new_pred = np.round(bst.predict(dtest_new))
acc_test_new = np.mean(y_test_new_pred == np.array(y_test_Mou))
roc_auc_new = roc_auc_score(np.array(y_test_Mou), bst.predict(dtest_new))
mcc = matthews_corrcoef(np.array(y_test_Mou), y_test_new_pred)

print("Accuracy on test set: %s, ROC-AUC score for test set: %s, MCC: %s"  % (acc_test_new, roc_auc_new, mcc))


[0]	eval-error:0.45833	train-error:0.23850
[1]	eval-error:0.47917	train-error:0.20712
[2]	eval-error:0.43750	train-error:0.19686
[3]	eval-error:0.39583	train-error:0.18801
[4]	eval-error:0.41667	train-error:0.17937
[5]	eval-error:0.41667	train-error:0.16908
[6]	eval-error:0.35417	train-error:0.16506
[7]	eval-error:0.39583	train-error:0.15506
[8]	eval-error:0.33333	train-error:0.14022
[9]	eval-error:0.33333	train-error:0.13710
[10]	eval-error:0.35417	train-error:0.13389
[11]	eval-error:0.35417	train-error:0.12548
[12]	eval-error:0.35417	train-error:0.11739
[13]	eval-error:0.37500	train-error:0.11145
[14]	eval-error:0.37500	train-error:0.10758
[15]	eval-error:0.37500	train-error:0.10346
[16]	eval-error:0.37500	train-error:0.09918
[17]	eval-error:0.29167	train-error:0.09587
[18]	eval-error:0.29167	train-error:0.09274
[19]	eval-error:0.31250	train-error:0.08711
[20]	eval-error:0.31250	train-error:0.08513
[21]	eval-error:0.31250	train-error:0.08308
[22]	eval-error:0.33333	train-error:0.0814

### Adding no Mou et al. data to the training set:

In [6]:

df_train = pd.read_pickle(join(CURRENT_DIR, ".." ,"data",
                               "splits", "df_train_with_ESM1b_ts.pkl"))
df_train = df_train.loc[df_train["ESM1b_ts"] != ""]
df_train.reset_index(inplace = True, drop = True)


train_X, train_y =  create_input_and_output_data(df = df_train)
test_X, test_y =  create_input_and_output_data(df = df_test)

df_test_new = df_Mou.copy()
df_test_new["Binding"] = [y > 2 for y in df_test_new["activity"]]
test_new_X, test_new_y =  create_input_and_output_data(df = df_test_new)


from sklearn.model_selection import train_test_split
#same split as in Mou et al paper:
X_train_Mou, X_test_Mou, y_train_Mou, y_test_Mou = train_test_split(test_new_X, test_new_y,
                                                                    test_size = 0.20, random_state = 888)




param = {'learning_rate': 0.31553117247348733,
         'max_delta_step': 1.7726044219753656,
         'max_depth': 10,
         'min_child_weight': 1.3845040588450772,
         'num_rounds': 342.68325188584106,
         'reg_alpha': 0.531395259755843,
         'reg_lambda': 3.744980563764689,
         'weight': 0.26187490421514203}



num_round = param["num_rounds"]
#param["tree_method"] = "gpu_hist"
#param["sampling_method"] = "gradient_based"
param['objective'] = 'binary:logistic'

weights =  np.array([param["weight"] if binding == 0 else 1.0 for binding in np.array(train_y)])

del param["num_rounds"]
del param["weight"]

dtrain = xgb.DMatrix(np.array(train_X), weight = weights, label = np.array(train_y),
            feature_names= feature_names)
dtest = xgb.DMatrix(np.array(test_X), label = np.array(test_y),
                    feature_names= feature_names)

dtest_new = xgb.DMatrix(np.array(X_test_Mou), label = np.array(y_test_Mou), feature_names= feature_names)

evallist = [(dtest_new, 'eval'), (dtrain, 'train')]

bst = xgb.train(param,  dtrain, int(num_round),evallist, verbose_eval=1)
y_test_pred = np.round(bst.predict(dtest))
acc_test = np.mean(y_test_pred == np.array(test_y))
roc_auc = roc_auc_score(np.array(test_y), bst.predict(dtest))

print("Accuracy on test set: %s, ROC-AUC score for test set: %s"  % (acc_test, roc_auc))


y_test_new_pred = np.round(bst.predict(dtest_new))
acc_test_new = np.mean(y_test_new_pred == np.array(y_test_Mou))
roc_auc_new = roc_auc_score(np.array(y_test_Mou), bst.predict(dtest_new))
mcc = matthews_corrcoef(np.array(y_test_Mou), y_test_new_pred)

print("Accuracy on test set: %s, ROC-AUC score for test set: %s, MCC: %s"  % (acc_test_new, roc_auc_new, mcc))


/home/hanxudong/miniconda3/envs/esp/lib/python3.8/site-packages/pandas/core/ops/array_ops.py:82: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  result = libops.scalar_compare(x.ravel(), y, op)


[0]	eval-error:0.47917	train-error:0.23690
[1]	eval-error:0.43750	train-error:0.20535
[2]	eval-error:0.39583	train-error:0.19503
[3]	eval-error:0.39583	train-error:0.18490
[4]	eval-error:0.43750	train-error:0.17935
[5]	eval-error:0.41667	train-error:0.16887
[6]	eval-error:0.45833	train-error:0.15759
[7]	eval-error:0.43750	train-error:0.15324
[8]	eval-error:0.43750	train-error:0.14741
[9]	eval-error:0.45833	train-error:0.13745
[10]	eval-error:0.45833	train-error:0.12460
[11]	eval-error:0.45833	train-error:0.11604
[12]	eval-error:0.43750	train-error:0.11029
[13]	eval-error:0.43750	train-error:0.10616
[14]	eval-error:0.45833	train-error:0.09944
[15]	eval-error:0.43750	train-error:0.09461
[16]	eval-error:0.43750	train-error:0.09130
[17]	eval-error:0.37500	train-error:0.08820
[18]	eval-error:0.37500	train-error:0.08595
[19]	eval-error:0.39583	train-error:0.08209
[20]	eval-error:0.39583	train-error:0.07924
[21]	eval-error:0.39583	train-error:0.07627
[22]	eval-error:0.39583	train-error:0.0746